# LAB 4: Pipeline de Ingestão em Escala
## Aula 5 — Docling e Ingestão Inteligente de Documentos
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração estimada:** 55 minutos  
**📚 Pré-requisitos:** Labs 1, 2 e 3 concluídos  
**🎯 Objetivos:**
- Implementar processamento paralelo de múltiplos documentos com ThreadPoolExecutor
- Criar sistema de cache em disco para evitar reprocessamento
- Implementar pipeline completo: Docling → Chunking → Metadados
- Gerar embeddings BGE-M3 em batch
- Indexar no OpenSearch (ou fallback FAISS)

In [ ]:
!pip install docling>=2.0.0 reportlab sentence-transformers faiss-cpu --quiet
print("✅ Instalação concluída!")

In [ ]:
import sys, warnings, os, json, re, hashlib, pickle, time
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

print("✅ Ambiente configurado")

## 📁 Etapa 1: Criar Corpus de Documentos para Ingestão em Lote

Vamos criar um mini-corpus de 5 documentos jurídicos distintos para
simular um cenário de ingestão em escala.

**⏱️ Tempo estimado: 8 minutos**

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib.enums import TA_JUSTIFY
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer

CORPUS_DIR = Path('/tmp/corpus_juridico_aula5')
CORPUS_DIR.mkdir(exist_ok=True)

def criar_pdf_simples(caminho, titulo, texto_corpo):
    # Cria um PDF simples com titulo e corpo de texto
    doc = SimpleDocTemplate(str(caminho), pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    st_titulo = ParagraphStyle('t', parent=styles['Heading1'], fontSize=13)
    st_corpo = ParagraphStyle('c', parent=styles['Normal'],
        fontSize=10, alignment=TA_JUSTIFY, leading=14)
    elementos = [
        Paragraph(titulo, st_titulo),
        Spacer(1, 0.5*cm),
        Paragraph(texto_corpo, st_corpo)
    ]
    doc.build(elementos)

# Criar 5 documentos diferentes
documentos = [
    {
        'arquivo': 'hc_stj_001.pdf',
        'titulo': 'Habeas Corpus nº 892.341-SP — STJ',
        'texto': ('HABEAS CORPUS. TRÁFICO DE ENTORPECENTES. ART. 33 LEI 11.343/2006. '
                  'PRISÃO PREVENTIVA. FUNDAMENTAÇÃO INIDÔNEA. GRAVIDADE ABSTRATA DO DELITO. '
                  'INSUFICIÊNCIA. A jurisprudência consolidada do STJ exige fundamentação '
                  'concreta e individualizada para a prisão preventiva. A simples referência '
                  'à gravidade do crime não basta. Súmula 440/STJ. ORDEM CONCEDIDA.')
    },
    {
        'arquivo': 'sumulas_stj_penal.pdf',
        'titulo': 'Súmulas STJ — Direito Penal (Seleção)',
        'texto': ('SÚMULA 231: A incidência da atenuante não pode reduzir a pena abaixo do '
                  'mínimo legal. SÚMULA 440: Fixada a pena-base no mínimo legal, é vedado '
                  'regime mais gravoso apenas pela gravidade abstrata do delito. '
                  'SÚMULA 444: É vedada a utilização de inquéritos policiais e ações penais '
                  'em curso para agravar a pena-base. SÚMULA 512: A aplicação do art. 33 §4º '
                  'Lei 11.343/2006 não afasta a hediondez do tráfico.')
    },
    {
        'arquivo': 'lgpd_segpublica.pdf',
        'titulo': 'LGPD e Segurança Pública — Aspectos Relevantes',
        'texto': ('A Lei 13.709/2018 (LGPD) estabelece regime especial para dados pessoais '
                  'sensíveis (art. 5º, II). Imagens de videomonitoramento policial são dados '
                  'sensíveis. O Decreto 68.447/2024-SP regulamenta a retenção de imagens '
                  'por 30 dias. A LGPD permite o tratamento de dados pessoais pela '
                  'administração pública para fins de segurança pública (art. 7º, III). '
                  'O STF (RE 1.055.941, Tema 990) autorizou o compartilhamento de dados '
                  'do COAF sem autorização judicial prévia, com comunicação posterior ao juiz.')
    },
    {
        'arquivo': 'lei_maria_penha.pdf',
        'titulo': 'Lei Maria da Penha (Lei 11.340/2006) — Aspectos Processuais',
        'texto': ('A Lei 11.340/2006 cria mecanismos de coibição e prevenção à violência '
                  'doméstica contra a mulher. O art. 24-A tipifica o descumprimento de '
                  'medida protetiva de urgência com pena de 3 meses a 2 anos de detenção. '
                  'O crime é formal: consuma-se com a mera inobservância da medida judicial. '
                  'O consentimento da vítima não afasta a tipicidade (TJ-SP, APL 1023456). '
                  'Medidas protetivas: afastamento do lar, proibição de contato, '
                  'monitoração eletrônica (art. 22).')
    },
    {
        'arquivo': 'licitacao_emergencia.pdf',
        'titulo': 'Contratação Direta por Emergência — Lei 14.133/2021',
        'texto': ('A Lei 14.133/2021 (NLLC) regula licitações e contratos administrativos. '
                  'O art. 75, VIII dispensa licitação em casos de emergência quando houver '
                  'risco de prejuízo à continuidade dos serviços públicos ou segurança de '
                  'pessoas. Requisitos: (1) comprovação documental da emergência; '
                  '(2) limitação ao estritamente necessário; (3) prazo máximo de execução '
                  'de 1 ano. A PGE-SP (Parecer 2024-0234) recomenda mapa de preços e '
                  'justificativa fundamentada da escolha do fornecedor.')
    },
]

for doc_info in documentos:
    caminho = CORPUS_DIR / doc_info['arquivo']
    criar_pdf_simples(caminho, doc_info['titulo'], doc_info['texto'])

print(f"✅ {len(documentos)} documentos criados em {CORPUS_DIR}")
for d in documentos:
    p = CORPUS_DIR / d['arquivo']
    print(f"   • {d['arquivo']} ({os.path.getsize(p)/1024:.1f} KB)")

## ⚡ Etapa 2: Processamento Paralelo com ThreadPoolExecutor

Processar documentos em paralelo reduz significativamente o tempo de ingestão.
Vamos comparar o processamento sequencial vs. paralelo.

**⏱️ Tempo estimado: 15 minutos**

In [ ]:
def processar_documento(caminho_pdf, converter):
    # Processa um único documento e retorna resultado com metadados
    inicio = time.time()
    try:
        resultado = converter.convert(str(caminho_pdf))
        doc = resultado.document
        markdown = doc.export_to_markdown()

        return {
            'arquivo': caminho_pdf.name,
            'status': 'sucesso',
            'paginas': len(doc.pages),
            'chars': len(markdown),
            'markdown': markdown,
            'tempo_s': round(time.time() - inicio, 2)
        }
    except Exception as e:
        return {
            'arquivo': caminho_pdf.name,
            'status': 'erro',
            'erro': str(e),
            'tempo_s': round(time.time() - inicio, 2)
        }

arquivos = list(CORPUS_DIR.glob('*.pdf'))
print(f"📂 {len(arquivos)} documentos para processar\n")

# --- Processamento SEQUENCIAL (baseline)
converter = DocumentConverter()
inicio_seq = time.time()
resultados_seq = []
for arq in arquivos:
    r = processar_documento(arq, converter)
    resultados_seq.append(r)
    emoji = "✅" if r["status"] == "sucesso" else "❌"
    print(f"  {emoji} {r['arquivo']} — {r['chars']} chars ({r['tempo_s']}s)")

tempo_seq = time.time() - inicio_seq
print(f"\n⏱️ Tempo sequencial: {tempo_seq:.1f}s")

# --- Processamento PARALELO
inicio_par = time.time()
resultados_par = []
print("\n🚀 Iniciando processamento paralelo (4 workers)...")

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(processar_documento, arq, DocumentConverter()): arq
               for arq in arquivos}
    for future in as_completed(futures):
        r = future.result()
        resultados_par.append(r)
        emoji = "✅" if r["status"] == "sucesso" else "❌"
        print(f"  {emoji} {r['arquivo']} — {r['chars']} chars ({r['tempo_s']}s)")

tempo_par = time.time() - inicio_par
print(f"\n⏱️ Tempo paralelo: {tempo_par:.1f}s")
print(f"⚡ Speedup: {tempo_seq/tempo_par:.1f}x")

## 💾 Etapa 3: Cache em Disco para Evitar Reprocessamento

Em produção, documentos raramente mudam. Um sistema de cache evita
reprocessar documentos já convertidos, economizando tempo e recursos.

**⏱️ Tempo estimado: 10 minutos**

In [ ]:
class CacheDocling:
    # Cache em disco para evitar reprocessamento de documentos ja convertidos
    def __init__(self, cache_dir='/tmp/docling_cache'):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.hits = 0
        self.misses = 0

    def _hash(self, caminho):
        # Hash MD5 do conteudo do arquivo
        md5 = hashlib.md5()
        with open(caminho, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                md5.update(chunk)
        return md5.hexdigest()

    def obter(self, caminho):
        h = self._hash(caminho)
        cache_path = self.cache_dir / f'{h}.pkl'
        if cache_path.exists():
            self.hits += 1
            with open(cache_path, 'rb') as f:
                return pickle.load(f)
        self.misses += 1
        return None

    def salvar(self, caminho, resultado):
        h = self._hash(caminho)
        cache_path = self.cache_dir / f'{h}.pkl'
        with open(cache_path, 'wb') as f:
            pickle.dump(resultado, f)

    def stats(self):
        total = self.hits + self.misses
        taxa = self.hits/total*100 if total > 0 else 0
        return {'hits': self.hits, 'misses': self.misses, 'hit_rate': f'{taxa:.0f}%'}

# Testar o cache
cache = CacheDocling()
converter_cache = DocumentConverter()

def processar_com_cache(caminho, converter, cache_obj):
    # Tenta buscar do cache, processa se nao encontrado
    resultado = cache_obj.obter(caminho)
    if resultado is not None:
        print(f'  🗃️  {caminho.name} — CACHE HIT')
        return resultado
    resultado = processar_documento(caminho, converter)
    if resultado['status'] == 'sucesso':
        cache_obj.salvar(caminho, resultado)
    print(f"  💾 {caminho.name} — PROCESSADO E CACHEADO ({resultado['tempo_s']}s)")
    return resultado

# Primeira execução (todos vão ao disco)
print("=== PRIMEIRA EXECUÇÃO (cache frio) ===")
for arq in arquivos:
    processar_com_cache(arq, converter_cache, cache)
print(f"Stats: {cache.stats()}")

# Segunda execução (todos virão do cache)
print("\n=== SEGUNDA EXECUÇÃO (cache quente) ===")
inicio_cache = time.time()
for arq in arquivos:
    processar_com_cache(arq, converter_cache, cache)
print(f"\n⚡ Tempo com cache: {time.time()-inicio_cache:.2f}s")
print(f"📊 Stats: {cache.stats()}")

## 🔗 Etapa 4: Pipeline Completo — Docling → Chunks → Embeddings BGE-M3

Agora vamos montar o pipeline de ponta a ponta: ingestão, chunking, e embeddings.

**⏱️ Tempo estimado: 15 minutos**

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("⏳ Carregando modelo BGE-M3 (pode demorar 1-2 min no Colab)...")
modelo = SentenceTransformer('BAAI/bge-m3')
print("✅ Modelo BGE-M3 carregado")

chunker = HybridChunker(tokenizer='BAAI/bge-m3', max_tokens=400, merge_peers=True)

def pipeline_ingestao_completo(arquivos, converter, chunker, modelo, cache=None):
    # Pipeline completo: Docling -> chunks -> embeddings
    todos_chunks = []

    for caminho in arquivos:
        # 1. Converter documento
        if cache:
            res = processar_com_cache(caminho, converter, cache)
        else:
            res = processar_documento(caminho, converter)

        if res['status'] != 'sucesso':
            print(f'  ⚠️ Pulando {caminho.name}: {res.get("erro")}')
            continue

        # 2. Chunking
        resultado_conv = converter.convert(str(caminho))
        chunks_doc = list(chunker.chunk(resultado_conv.document))

        # 3. Criar objetos de chunk com metadados
        for i, chunk in enumerate(chunks_doc):
            if len(chunk.text.strip()) < 30:  # Filtrar chunks muito pequenos
                continue
            todos_chunks.append({
                'texto': chunk.text,
                'arquivo': caminho.name,
                'chunk_index': i,
                'num_chars': len(chunk.text)
            })

    print(f"\n✅ Total de chunks: {len(todos_chunks)}")

    # 4. Gerar embeddings em batch
    textos = [c['texto'] for c in todos_chunks]
    print(f"\n⏳ Gerando embeddings BGE-M3 para {len(textos)} chunks...")

    embeddings = modelo.encode(
        textos,
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    # 5. Adicionar embeddings aos chunks
    for i, chunk in enumerate(todos_chunks):
        chunk['embedding'] = embeddings[i].tolist()

    print(f"\n✅ Embeddings gerados: shape={embeddings.shape}")
    return todos_chunks, embeddings

# Executar pipeline
cache2 = CacheDocling('/tmp/docling_cache_v2')
chunks_finais, embeddings_np = pipeline_ingestao_completo(
    arquivos, DocumentConverter(), chunker, modelo, cache=cache2
)

In [ ]:
# Indexar com FAISS (fallback quando OpenSearch nao disponivel)
import faiss

USE_OPENSEARCH = False  # Mude para True se OpenSearch estiver disponivel

if USE_OPENSEARCH:
    print('TODO: indexar no OpenSearch (veja Lab 5)')
else:
    DIM = embeddings_np.shape[1]  # 1024 para BGE-M3
    index = faiss.IndexFlatIP(DIM)  # Inner Product = cosine com vetores normalizados
    index.add(embeddings_np.astype('float32'))
    print(f"⚠️ Usando FAISS local (sem OpenSearch)")
    print(f"✅ FAISS index criado: {index.ntotal} vetores, dim={DIM}")

# Salvar índice
faiss.write_index(index, '/tmp/aula5_corpus.faiss')

# Salvar metadados
chunks_sem_embedding = [{k: v for k, v in c.items() if k != 'embedding'}
                        for c in chunks_finais]
with open('/tmp/aula5_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_sem_embedding, f, ensure_ascii=False, indent=2)

print(f"\n✅ Índice FAISS salvo: /tmp/aula5_corpus.faiss")
print(f"✅ Metadados salvos: /tmp/aula5_chunks.json ({len(chunks_sem_embedding)} chunks)")

## ✅ Checkpoint — Lab 4


In [ ]:
print("=" * 60)
print("CHECKPOINT — LAB 4")
print("=" * 60)
erros = []

if len(resultados_par) == 0:
    erros.append('❌ Processamento paralelo não executado')
else:
    print(f"✅ {len(resultados_par)} documentos processados em paralelo")

if cache.stats()['hits'] == 0:
    erros.append('❌ Cache não funcionou')
else:
    print(f"✅ Cache funcionando: {cache.stats()}")

if len(chunks_finais) == 0:
    erros.append('❌ Pipeline completo não gerou chunks')
else:
    print(f"✅ Pipeline completo: {len(chunks_finais)} chunks com embeddings")

if not Path('/tmp/aula5_corpus.faiss').exists():
    erros.append('❌ Índice FAISS não criado')
else:
    print("✅ Índice FAISS criado")

if erros:
    for e in erros: print(e)
else:
    print("\n🎉 TODOS OS OBJETIVOS ATINGIDOS! Avance para o Lab 5.")

## 🏋️ Exercício

**Desafio:** Implemente uma busca simples no índice FAISS criado:
1. Faça uma query: `'quais os requisitos para prisão preventiva?'`
2. Gere o embedding da query com BGE-M3
3. Busque os top-5 chunks mais similares
4. Imprima o texto de cada resultado com seu score


In [ ]:
# 🏋️ EXERCÍCIO: Busca simples no índice FAISS
query = 'quais os requisitos para prisao preventiva?'

# Gerar embedding da query
query_embedding = modelo.encode([query], normalize_embeddings=True).astype('float32')

# Buscar top-5
D, I = index.search(query_embedding, 5)

print(f"Query: {query}\n")
print("Top-5 resultados:")
for rank, (score, idx) in enumerate(zip(D[0], I[0])):
    chunk = chunks_sem_embedding[idx]
    print(f"\n{rank+1}. Score: {score:.3f} | Arquivo: {chunk['arquivo']}")
    print(f"   Texto: {chunk['texto'][:200]}...")

## 📖 Referências (ABNT)

AUER, Peter et al. **Docling Technical Report**. arXiv:2408.09869, 2024.

CHEN, J. et al. **BGE M3-Embedding**. arXiv:2309.07597, 2024.

JOHNSON, Jeff; DOUZE, Matthijs; JÉGOU, Hervé. **Billion-scale Similarity Search with GPUs**.
*IEEE Transactions on Big Data*, v. 7, n. 3, p. 535-547, 2021.
